In [2]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav.columns


Index(['amfi_code', 'date', 'nav'], dtype='str')

In [7]:
nav.dtypes

amfi_code      int64
date             str
nav          float64
dtype: object

In [8]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [9]:
nav.duplicated().sum()

np.int64(0)

In [11]:
nav["date"] = pd.to_datetime(nav["date"])

In [12]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [13]:
nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
       .ffill()
)


In [14]:
nav = nav.drop_duplicates()

In [15]:
nav.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

In [19]:
nav.columns

Index(['amfi_code', 'date', 'nav'], dtype='str')

In [20]:
nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [22]:
txn = pd.read_csv("../data/raw/08_investor_transactions.csv")

txn.columns

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='str')

In [23]:
txn["transaction_type"] = (
    txn["transaction_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

txn["transaction_type"] = txn["transaction_type"].replace({
    "SIP PAYMENT":"SIP",
    "SIP":"SIP",
    "LUMPSUM":"LUMPSUM",
    "REDEEM":"REDEMPTION",
    "REDEMPTION":"REDEMPTION"
})

In [26]:
invalid_amount = txn[txn["amount_inr"] <= 0]

print("Invalid Amount Records:", len(invalid_amount))

Invalid Amount Records: 0


In [27]:
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"],
    errors="coerce"
)

In [28]:
txn["kyc_status"] = txn["kyc_status"].str.upper()

valid_kyc = ["YES","NO","PENDING"]

invalid_kyc = txn[
    ~txn["kyc_status"].isin(valid_kyc)
]

print("Invalid KYC:", len(invalid_kyc))

Invalid KYC: 30146


In [29]:
txn.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

In [31]:
perf = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

perf.columns

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')

In [35]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

In [36]:
for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

In [37]:
for col in return_cols:
    anomalies = perf[
        perf[col].abs() > 100
    ]

    print(col, len(anomalies))

return_1yr_pct 0
return_3yr_pct 0
return_5yr_pct 0


In [39]:
bad_expense = perf[
    (perf["expense_ratio_pct"] < 0.1)
    |
    (perf["expense_ratio_pct"] > 2.5)
]

print("Bad Expense Ratio:", len(bad_expense))

Bad Expense Ratio: 0


In [40]:
perf.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

In [41]:
from sqlalchemy import create_engine
import pandas as pd

# Create SQLite database
engine = create_engine(
    "sqlite:///mutual_fund_analytics.db"
)

In [42]:
nav = pd.read_csv(
    "../data/processed/nav_history_clean.csv"
)

nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [43]:
nav.to_sql(
    "nav_history",
    engine,
    if_exists="replace",
    index=False
)

print("NAV table loaded successfully")

NAV table loaded successfully


In [44]:
import sqlite3

conn = sqlite3.connect(
    "mutual_fund_analytics.db"
)

query = """
SELECT COUNT(*)
FROM nav_history
"""

print(
    pd.read_sql(query, conn)
)

   COUNT(*)
0     46000


In [45]:
txn = pd.read_csv(
    "../data/processed/investor_transactions_clean.csv"
)

txn.to_sql(
    "investor_transactions",
    engine,
    if_exists="replace",
    index=False
)

32778

In [46]:
perf = pd.read_csv(
    "../data/processed/scheme_performance_clean.csv"
)

perf.to_sql(
    "scheme_performance",
    engine,
    if_exists="replace",
    index=False
)

40

In [47]:
txn = pd.read_csv("../data/processed/investor_transactions_clean.csv")

txn.to_sql(
    "investor_transactions",
    engine,
    if_exists="replace",
    index=False
)

print("investor_transactions loaded")

investor_transactions loaded


In [48]:
perf = pd.read_csv("../data/processed/scheme_performance_clean.csv")

perf.to_sql(
    "scheme_performance",
    engine,
    if_exists="replace",
    index=False
)

print("scheme_performance loaded")

scheme_performance loaded


In [49]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mutual_fund_analytics.db")

tables = [
    "nav_history",
    "investor_transactions",
    "scheme_performance"
]

for table in tables:
    query = f"SELECT COUNT(*) as rows FROM {table}"
    print(table)
    print(pd.read_sql(query, conn))
    print("-" * 30)

nav_history
    rows
0  46000
------------------------------
investor_transactions
    rows
0  32778
------------------------------
scheme_performance
   rows
0    40
------------------------------
